In [3]:
pip install --upgrade typing_extensions

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageOps
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import torchvision.transforms as transforms

# ----------------------------------------------------
# 步驟 1：定義 Pipeline A 資料集 (PyTorch 規格)
# ----------------------------------------------------
class BreastCancerDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # 讀取影像
        img_path = self.image_paths[idx]
        img = Image.open(img_path).convert('RGB')
        
        # Pipeline A：補黑邊轉正方形，避免腫瘤變形
        w, h = img.size
        max_side = max(w, h)
        padding = ((max_side - w) // 2, (max_side - h) // 2, 
                   (max_side - w) - (max_side - w) // 2, (max_side - h) - (max_side - h) // 2)
        img = ImageOps.expand(img, padding, fill=0)
        img = img.resize((224, 224))
        
        # 標籤與基礎轉換
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        if self.transform:
            img = self.transform(img)
            
        return img, label

In [5]:
# ----------------------------------------------------
# 步驟 2：搜尋本地硬碟資料並分割
# ----------------------------------------------------
path = "C:/Users/謝秉岑/.cache/kagglehub/datasets/dynemiesizumaki/cbis-ddsmpng/versions/1"

all_image_paths = []
all_labels = []

print("正在掃描硬碟影像檔案...")
for root, dirs, files in os.walk(path):
    if "roi_cropped.png" in files:
        # 良性 0, 惡性 1
        label = 1 if "MALIGNANT" in root.upper() else 0
        all_image_paths.append(os.path.join(root, "roi_cropped.png"))
        all_labels.append(label)

# 分割資料集 (80% 訓練 + 驗證, 20% 測試)
train_paths, test_paths, train_labels, test_labels = train_test_split(
    all_image_paths, all_labels, test_size=0.2, stratify=all_labels, random_state=42
)
# 再從訓練中切出 10% 做驗證 (模擬考)
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_paths, train_labels, test_size=0.1, stratify=train_labels, random_state=42
)

# 轉換器：轉為 Tensor 並歸一化 (Normalization)
data_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # ImageNet 標準歸一化
])

# 建立 DataLoader 產生器
train_loader = DataLoader(BreastCancerDataset(train_paths, train_labels, data_transforms), batch_size=32, shuffle=True)
val_loader = DataLoader(BreastCancerDataset(val_paths, val_labels, data_transforms), batch_size=32, shuffle=False)
test_loader = DataLoader(BreastCancerDataset(test_paths, test_labels, data_transforms), batch_size=32, shuffle=False)

正在掃描硬碟影像檔案...


In [9]:
# ----------------------------------------------------
# 步驟 3：載入預訓練 ResNet50 並修改成二分類輸出
# ----------------------------------------------------
from torchvision.models import resnet50, ResNet50_Weights

print("正在載入官方預訓練 ResNet50...")
# 使用妳剛剛貼出的代碼中對應的官方最新權重格式
weights = ResNet50_Weights.DEFAULT
model = resnet50(weights=weights)

# 微調策略：凍結前段，解凍後段 (最後 20 層)
for param in model.parameters():
    param.requires_grad = False

# 解凍最後一組卷積層 (layer4) 與分類頭，強制它學習 X 光片特徵
for param in model.layer4.parameters():
    param.requires_grad = True

# 修改原本的 FC 層（1000類輸出）改成我們的二分類輸出（良性 vs 惡性）
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_ftrs, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 1) # 輸出 1 個數值，後面搭配 BCEWithLogitsLoss
)

# 檢查是否有 GPU 加速 (CUDA)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# ----------------------------------------------------
# 步驟 4：設定損失函數與極小的學習率 (微微調)
# ----------------------------------------------------
criterion = nn.BCEWithLogitsLoss() # 整合了 Sigmoid 的二分類損失函數
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)

# ----------------------------------------------------
# 步驟 5：開始訓練迴圈 (跑 20 輪)
# ----------------------------------------------------
epochs = 20
print(f"環境就緒：使用 {device} 啟動 PyTorch 訓練...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {epoch_loss:.4f}")

# ----------------------------------------------------
# 步驟 6：最終期末考（丟入測試集產出成績單）
# ----------------------------------------------------
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs).squeeze()
        
        # 通過算出來的機率進行二分類判斷 (>0.5)
        preds = (torch.sigmoid(outputs) > 0.46).cpu().numpy().astype(int)
        all_preds.extend(preds)
        all_targets.extend(labels.numpy().astype(int))

print("\n" + "="*40)
print("【PyTorch 官方規格 ResNet50 測試集最終報表】")
print(classification_report(all_targets, all_preds, target_names=['Benign', 'Malignant']))
print("="*40)

正在載入官方預訓練 ResNet50...
環境就緒：使用 cpu 啟動 PyTorch 訓練...
Epoch 1/20 - Train Loss: 0.6888
Epoch 2/20 - Train Loss: 0.6886
Epoch 3/20 - Train Loss: 0.6833
Epoch 4/20 - Train Loss: 0.6811
Epoch 5/20 - Train Loss: 0.6812
Epoch 6/20 - Train Loss: 0.6786
Epoch 7/20 - Train Loss: 0.6718
Epoch 8/20 - Train Loss: 0.6680
Epoch 9/20 - Train Loss: 0.6669
Epoch 10/20 - Train Loss: 0.6624
Epoch 11/20 - Train Loss: 0.6622
Epoch 12/20 - Train Loss: 0.6569
Epoch 13/20 - Train Loss: 0.6548
Epoch 14/20 - Train Loss: 0.6447
Epoch 15/20 - Train Loss: 0.6454
Epoch 16/20 - Train Loss: 0.6391
Epoch 17/20 - Train Loss: 0.6327
Epoch 18/20 - Train Loss: 0.6281
Epoch 19/20 - Train Loss: 0.6222
Epoch 20/20 - Train Loss: 0.6075

【PyTorch 官方規格 ResNet50 測試集最終報表】
              precision    recall  f1-score   support

      Benign       0.69      0.81      0.74        57
   Malignant       0.66      0.50      0.57        42

    accuracy                           0.68        99
   macro avg       0.67      0.65      0.65    